# Calcualting aggregate statistic values

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

In [2]:
DATA_PATH = 'reports'
ALL_DATA = os.path.join(DATA_PATH, 'all.parquet')
print(os.path.exists(ALL_DATA))
COEF_VAR = 'b'

True


In [3]:
df = pd.read_parquet(ALL_DATA)
df.shape

(42286, 7)

In [4]:
df.head()

,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,True,0.509592,-0.103579,10
1,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.928928,-1.726076,9
2,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.028320,0.307327,43
3,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,0.040052,-0.113704,41
4,CABNC,CABNC-CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUN,False,0.000211,3.142344,6


## Merging data

In [5]:
df = pd.merge(
    left=df.loc[df['self']], left_on=['cid', 'speaker'],
    right=df[['cid', 'speaker', COEF_VAR, 'a']].loc[~df['self']], right_on=['cid', 'speaker'],
    how='inner'
)
df.shape

(26359, 9)

In [6]:
df.isna().sum()

corpus     0
cid        0
speaker    0
self       0
a_x        0
b_x        0
k          0
b_y        0
a_y        0
dtype: int64

In [7]:
# df = df.loc[~df['b_y'].isna()]

In [8]:
df['b_delta'] = df['b_y'] - df['b_x']
# df['b_delta'] = np.log(np.abs( np.exp(df['b_x']) - np.exp(df['b_y']) ))
# df['b_delta'] = np.log(
#     np.abs(
#         np.exp(df['b_x'] + df['a_x']) - np.exp(df['b_y'] + df['a_y']) + 1e-9
#     )
# )


## T-statistics

### Difference from zero

In [9]:
null = 0

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values

dof = len(vs)-1

tstat = (vs.mean() - null) / (vs.std() / np.sqrt(len(vs)-1))

# tstat = ((df0[COEF_VAR] - null) / df0['se']).values
vs.mean(), vs.std(), dof, tstat, t.sf(tstat.__abs__(), df=dof)

(np.float64(0.04992622658139719),
 np.float64(0.9130807476726379),
 26358,
 np.float64(8.87719550186465),
 np.float64(3.6413385958686376e-19))

### Difference from identity

In [10]:
null = 1

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values

tstat = (vs.mean() - null) / (vs.std() / np.sqrt(len(vs)-1))

# tstat = ((df0[COEF_VAR] - null) / df0['se']).values
vs.mean(), vs.std(), tstat, t.sf(tstat.__abs__(), df=len(df) - 1)

(np.float64(0.04992622658139719),
 np.float64(0.9130807476726379),
 np.float64(-168.9290620447921),
 np.float64(0.0))

### Wilcoxan Signed Rank Test

Because I'm not sure I want to bet on the data being normally distributed...

In [11]:
from scipy.stats import wilcoxon

In [12]:
null = 0

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values - null

wilcoxon(vs)

WilcoxonResult(statistic=np.float64(155643236.0), pvalue=np.float64(2.076333642247684e-48))

In [12]:
null = 1

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values - null

wilcoxon(vs)

WilcoxonResult(statistic=np.float64(40230449.0), pvalue=np.float64(0.0))

## Visualizations!

In [13]:
import plotly.figure_factory as ff

In [14]:
hist_data = [
    # df['b_delta'].loc[
    #     (~df['self'])
    #     & (~np.isinf(df['b_delta']))
    #     & (~df['b_delta'].isna())
    # ].values,
    df['b_delta'].loc[
        df['self']
        & (~np.isinf(df['b_delta']))
        & (~df['b_delta'].isna())
    ].values,
]
# print(len(hist_data[0]),len(hist_data[1]),)

groups = [
    # 'other',
    'self',
]

colors = [
    # '#0504aa',
    '#FFC300',
]

In [15]:
fig = ff.create_distplot(hist_data,groups,show_hist=False,colors=colors)

In [16]:
fig.update_layout(template='ygridoff',showlegend=False)

In [24]:
# fig.show()

In [25]:
fig.write_html(os.path.join(DATA_PATH, 'CM-results.html'))